# StylePops — Kombin + A/B Üretimi (Colab)

**GPU gerekmez · torchao gerekmez · FashionCLIP modeli yüklenmez.**

FashionCLIP embedding'leri repoda önceden hesaplanmış olarak gelir
(`data/visual/clip_embeddings.npz`). Colab sadece bu önbelleği okur →
torchao/transformers kurulumu ve fabrika sıfırlaması **artık gerekmiyor**.

## Mac (bir kez)
```bash
python scripts/precompute_clip_cache.py      # embedding önbelleği (commit edilir)
python scripts/pack_colab_combo_bundle.py    # görselleri paketle → zip
```

## Colab (sırayla)
1. **Hücre 1** → repo çek + **commit no'sunu göster** (doğru sürümde misin kontrol et)
2. **Hücre 2** → minimal bağımlılık (Pillow)
3. **Hücre 3** → `colab_combo_bundle.zip` yükle
4. **Hücre 4** → paketi aç
5. **Hücre 5** → üret
6. **Hücre 6** → sonucu indir

> Not: Artık her şey Mac'te de saniyeler içinde çalışır:
> `OMP_NUM_THREADS=1 python scripts/generate_visual_combinations.py --ab-pairs 160`
> Colab yalnızca yedek yoldur.

In [ ]:
# ============================================================
#  HÜCRE 1 — REPO ÇEK + SÜRÜM (her zaman ÖNCE bunu çalıştır)
# ============================================================
# Sayfayı yenilediğinde bu hücreyi çalıştır → doğru commit'te
# olup olmadığını anında görürsün. Fabrika sıfırlaması gerekmez.
import os, shutil, subprocess
from pathlib import Path

PROJECT = Path('/content/StylePops_Modules')
os.chdir('/content')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '-q',
                'https://github.com/valueisinvalid/StylePops_Modules.git'], check=True)
os.chdir(PROJECT)
subprocess.run(['git', 'fetch', '-q', 'origin', 'main'], check=False)
subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=False)

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
msg = subprocess.check_output(['git', 'log', '-1', '--pretty=%s'], text=True).strip()
cache = PROJECT / 'data/visual/clip_embeddings.npz'
print('=' * 56)
print(f'  REPO HAZIR · commit: {commit}')
print(f'  son commit: {msg}')
print(f'  CLIP önbellek: {"VAR ✓" if cache.exists() else "YOK (renk moduna düşer)"}')
print('=' * 56)

In [ ]:
# ============================================================
#  HÜCRE 2 — Minimal bağımlılık (torchao / transformers YOK)
# ============================================================
# FashionCLIP embedding'leri repodaki önbellekten okunur,
# model yüklenmez → torchao sorunları tamamen ortadan kalkar.
!pip install -q -U --force-reinstall 'pillow>=10.4.0,<12' >/dev/null 2>&1

import PIL, numpy
from PIL import Image, ImageDraw
print('Pillow:', PIL.__version__, '· numpy:', numpy.__version__, '· import OK')

In [ ]:
# ============================================================
#  HÜCRE 3 — Görsel paketini yükle (colab_combo_bundle.zip)
# ============================================================
# Mac'te: python scripts/pack_colab_combo_bundle.py
from google.colab import files
from pathlib import Path
import os, shutil

PROJECT = Path('/content/StylePops_Modules')
os.chdir(PROJECT)
print('Yükleme klasörü:', PROJECT.resolve())

uploaded = files.upload()
if not uploaded:
    raise FileNotFoundError('Zip seçilmedi → outputs/colab_combo_bundle.zip')

zip_name = next(iter(uploaded))
ZIP_PATH = PROJECT / zip_name
alt = Path('/content') / zip_name
if not ZIP_PATH.exists() and alt.exists():
    shutil.move(str(alt), str(ZIP_PATH))
print('Yüklendi:', ZIP_PATH, f'({ZIP_PATH.stat().st_size // 1024 // 1024} MB)')

In [ ]:
# ============================================================
#  HÜCRE 4 — Paketi aç (görseller + lookups)
# ============================================================
import zipfile, shutil, json
from pathlib import Path

PROJECT = Path('/content/StylePops_Modules')
if 'ZIP_PATH' not in globals() or not Path(ZIP_PATH).exists():
    raise RuntimeError('Önce Hücre 3 (zip yükle) çalıştır')

with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall('/content')

SRC = Path('/content/stylepops_colab')
# Gardırop + görseller paketten gelir; clip_embeddings.npz repodan (commit'li) gelir
shutil.copy2(SRC / 'data/visual/garments_production.json',
             PROJECT / 'data/visual/garments_production.json')
shutil.copytree(SRC / 'data' / 'lookups', PROJECT / 'data' / 'lookups', dirs_exist_ok=True)
for folder in ('data/assets/garments', 'data/assets/fashion_product'):
    src_dir = SRC / folder
    if src_dir.exists():
        shutil.copytree(src_dir, PROJECT / folder, dirs_exist_ok=True)

n = json.loads((PROJECT / 'data/visual/garments_production.json').read_text())['count']
cache = PROJECT / 'data/visual/clip_embeddings.npz'
print(f'Gardırop: {n} parça · CLIP önbellek: {"VAR ✓" if cache.exists() else "YOK"}')

In [ ]:
# ============================================================
#  HÜCRE 5 — Kombin + A/B üret (CLIP önbellek · torchao yok)
# ============================================================
import os, sys, runpy
from pathlib import Path

PROJECT = Path('/content/StylePops_Modules')
os.chdir(PROJECT)
Path('data/assets/combos').mkdir(parents=True, exist_ok=True)
os.environ['PYTHONUNBUFFERED'] = '1'

for name in list(sys.modules):
    if name.split('.')[0] in ('stylepops_core', 'aesthetic_compatibility',
                                'inventory_loader', 'build_combo_collage',
                                'garment_eligibility', 'garment_gender'):
        del sys.modules[name]

# Önbellek varsa FashionCLIP skoru (model yüklenmez); yoksa --fast renk modu
use_fast = not (PROJECT / 'data/visual/clip_embeddings.npz').exists()
args = ['generate_visual_combinations.py', '--per-scenario', '40',
        '--ab-pairs', '160', '--n-candidates', '250']
if use_fast:
    args.append('--fast')
sys.argv = args
print('Komut:', ' '.join(args))
runpy.run_path('scripts/generate_visual_combinations.py', run_name='__main__')

In [ ]:
# ============================================================
#  HÜCRE 6 — Sonuçları indir (Mac'e)
# ============================================================
import zipfile
from pathlib import Path
from google.colab import files

root = Path('/content/StylePops_Modules')
out = Path('/content/stylepops_results.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in ('data/visual/combinations_visual.csv', 'data/visual/ab_pairs.csv'):
        zf.write(root / name, Path(name).name)

combos_zip = Path('/content/stylepops_combos.zip')
!cd /content/StylePops_Modules/data/assets/combos && zip -qr {combos_zip} AB*.jpg VC*.jpg

print('Mac kurulum:')
print('  unzip -o stylepops_results.zip -d data/visual/')
print('  unzip -o stylepops_combos.zip  -d data/assets/combos/')
files.download(str(out))
files.download(str(combos_zip))

In [ ]:
# (kullanılmıyor — akış Hücre 6'da biter)

In [ ]:
# (kullanılmıyor)